# Module 01: Tokenization & Truncation

**ECBS5200 Pre-Work** — Practical Deep Learning Engineering

In this notebook, you'll tokenize real consumer complaints and see exactly
what happens to text before a model ever touches it. No GPU needed — this
runs on CPU in under a minute.

## Setup

We need two libraries: `transformers` (for the tokenizer) and `datasets`
(to load the actual complaint data we'll use all semester).

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset

# Load the tokenizer for the model we'll use in the course
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

print(f"Vocabulary size: {tokenizer.vocab_size:,}")
print(f"Model max length: {tokenizer.model_max_length:,}")

## Load the dataset

This is the dataset we'll work with all semester: consumer complaints
submitted to the U.S. Consumer Financial Protection Bureau (CFPB).
Each complaint has a text description and an issue label.

In [ ]:
dataset = load_dataset("determined-ai/consumer_complaints_medium", split="train")

print(f"Number of complaints: {len(dataset):,}")
print(f"Columns: {dataset.column_names}")
print(f"\nFirst complaint:")
print(f"  Label: {dataset[0]['Issue']}")
print(f"  Text:  {dataset[0]['Consumer Complaint'][:200]}...")

## Your first tokenization

Let's tokenize a short complaint and see exactly what comes out.

In [ ]:
# A short, real complaint
short_complaint = "There are many mistakes appear in my report without my understanding."

# Tokenize it
tokens = tokenizer(short_complaint, max_length=128, truncation=True, padding="max_length")

print("=== Tokenizer output ===")
print(f"Keys: {list(tokens.keys())}")
print(f"\nInput IDs (first 20): {tokens['input_ids'][:20]}")
print(f"Attention mask (first 20): {tokens['attention_mask'][:20]}")
print(f"\nTotal length: {len(tokens['input_ids'])}")
print(f"Real tokens (non-padding): {sum(tokens['attention_mask'])}")

Notice: the sequence is 128 numbers long (our `max_length`), but only some
are real tokens. The rest are padding. The attention mask tells us which is which.

## Decoding: from numbers back to text

We can convert token IDs back to readable tokens to see the subword pieces.

In [ ]:
# Convert IDs back to token strings
token_strings = tokenizer.convert_ids_to_tokens(tokens['input_ids'])

# Show only the real tokens (not padding)
real_token_count = sum(tokens['attention_mask'])
real_tokens = token_strings[:real_token_count]

print("Tokens:")
for i, (tok, tid) in enumerate(zip(real_tokens, tokens['input_ids'][:real_token_count])):
    print(f"  Position {i:3d}: {tok:20s}  →  ID {tid}")

Notice the `[CLS]` at the start and `[SEP]` at the end — the tokenizer
added those automatically. Every input gets these.

## Subword splitting in action

Common words stay whole. Unusual words get split into subword pieces.
Let's see this with some financial terms.

In [ ]:
test_words = [
    "mortgage",
    "refinancing",
    "overpayment",
    "foreclosure",
    "Equifax",
    "XXXX",  # redaction marker in our dataset
    "unauthorized",
]

print("=== Subword splitting ===")
for word in test_words:
    toks = tokenizer.tokenize(word)
    print(f"  {word:20s} → {toks}")

ModernBERT uses BPE (byte pair encoding) tokenization. You'll notice a `Ġ`
character before some tokens — that just means "this token started with a space"
(i.e., it's the beginning of a new word). Tokens without `Ġ` continue
the previous word.

Notice `XXXX` — that's a redaction marker the CFPB uses to hide personal
information. You'll see it a lot in the data.

## Truncation: what gets lost

Let's find a long complaint and see what happens when it exceeds our
`max_length` of 128 tokens.

In [ ]:
# Find a complaint that will be truncated at 128 tokens
long_complaint = None
for i in range(len(dataset)):
    text = dataset[i]['Consumer Complaint']
    token_count = len(tokenizer.encode(text))
    if 150 < token_count < 200:  # long enough to truncate, not enormous
        long_complaint = text
        long_complaint_tokens = token_count
        long_complaint_label = dataset[i]['Issue']
        break

print(f"Found a complaint with {long_complaint_tokens} tokens (label: {long_complaint_label})")
print(f"\nFull text:\n{long_complaint}")

In [ ]:
# Tokenize WITHOUT truncation (to see the full thing)
full_tokens = tokenizer(long_complaint, truncation=False)
print(f"Full token count: {len(full_tokens['input_ids'])}")

# Tokenize WITH truncation at 128
truncated_tokens = tokenizer(long_complaint, max_length=128, truncation=True)
print(f"Truncated token count: {len(truncated_tokens['input_ids'])}")
print(f"Tokens lost: {len(full_tokens['input_ids']) - 128}")

# Decode both to see the difference
full_text = tokenizer.decode(full_tokens['input_ids'], skip_special_tokens=True)
truncated_text = tokenizer.decode(truncated_tokens['input_ids'], skip_special_tokens=True)

print(f"\n=== FULL TEXT ===\n{full_text}")
print(f"\n=== TRUNCATED TEXT (what the model sees) ===\n{truncated_text}")
print(f"\n=== LOST TEXT (the model NEVER sees this) ===\n{full_text[len(truncated_text):]}")

That lost text? The model has no idea it exists. It makes its prediction
based only on the first 128 tokens. This is truncation in action.

## How much truncation happens in our dataset?

Let's check: what fraction of complaints fit within 128 tokens?

In [ ]:
import matplotlib.pyplot as plt

# Tokenize a sample to check lengths (full dataset takes a while, sample is fine)
sample_size = 5000
lengths = []
for i in range(sample_size):
    text = dataset[i]['Consumer Complaint']
    token_count = len(tokenizer.encode(text))
    lengths.append(token_count)

# Stats
import numpy as np
lengths = np.array(lengths)
fits_in_128 = (lengths <= 128).mean() * 100

print(f"Sample size: {sample_size:,}")
print(f"Median tokens: {np.median(lengths):.0f}")
print(f"Mean tokens: {np.mean(lengths):.0f}")
print(f"Min: {lengths.min()}, Max: {lengths.max()}")
print(f"\nComplaints that fit in 128 tokens: {fits_in_128:.1f}%")
print(f"Complaints that get truncated: {100 - fits_in_128:.1f}%")

In [ ]:
# Plot the distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(lengths, bins=80, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(x=128, color='red', linestyle='--', linewidth=2, label='max_length = 128')
ax.axvline(x=np.median(lengths), color='orange', linestyle='--', linewidth=2, label=f'Median = {np.median(lengths):.0f}')
ax.set_xlabel('Number of tokens', fontsize=12)
ax.set_ylabel('Number of complaints', fontsize=12)
ax.set_title('Token length distribution of consumer complaints', fontsize=14)
ax.legend(fontsize=11)
ax.set_xlim(0, 500)
plt.tight_layout()
plt.show()

print(f"\nMost complaints are short. The red line is our cutoff.")
print(f"About {100 - fits_in_128:.0f}% of complaints lose some text to truncation.")

## Check yourself

Before moving to Module 02, make sure you can answer these:

1. Why do we use subword tokenization instead of splitting on spaces?
2. What are the `[CLS]` and `[SEP]` tokens for?
3. If a complaint is 200 tokens long and max_length is 128, what happens
   to the last 72 tokens?
4. What does the attention mask tell the model?
5. Why did we choose 128 instead of 512 for max_length?